In [11]:
from sn_rating_v3.helpers import input_path
import warnings
import pandas as pd

rating_input_file = input_path("v3_rating_input.xlsx")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
    with pd.ExcelFile(rating_input_file) as xls:
        df_fin = pd.read_excel(xls, sheet_name="fin_ratios", index_col=0)
        df_qual = pd.read_excel(xls, sheet_name="qual_factor", index_col=0)
        df_meta = pd.read_excel(xls, sheet_name="metadata")
        df_peers = pd.read_excel(xls, sheet_name="Peers_t0")

# Drop Unnamed columns just like in report.py
df_fin = df_fin.loc[:, ~df_fin.columns.str.contains("^Unnamed")]
df_qual = df_qual.loc[:, ~df_qual.columns.str.contains("^Unnamed")]


In [13]:
from sn_rating_v3.run_from_excel import _infer_time_labels

YEAR_T0, YEAR_T1, YEAR_T2 = _infer_time_labels(df_fin)


In [15]:
from sn_rating_v3.run_from_excel import run_v3_from_excel_with_bands

result = run_v3_from_excel_with_bands("v3_rating_input.xlsx")

ratio_rows = []
for r in result.ratio_log:
    name = r.get("Name")
    if not name or name == "peer_positioning":
        continue
    ratio_rows.append((name, r.get("Value")))


2026-03-11 19:24:56,400 - INFO - sn_rating_v3 - BandConfig: loaded families/directions for ratios: {'altman_z': 'altman', 'capex_dep': 'other', 'current_ratio': 'other', 'debt_capital': 'leverage', 'debt_ebitda': 'leverage', 'debt_equity': 'leverage', 'dscr': 'coverage', 'ebit_margin': 'profit', 'ebitda_margin': 'profit', 'fcf_debt': 'leverage_rev', 'ffo_debt': 'leverage_rev', 'fixed_charge_coverage': 'coverage', 'interest_coverage': 'coverage', 'nan': 'nan', 'net_debt_ebitda': 'leverage', 'quick_ratio': 'other', 'roa': 'profit', 'roe': 'profit', 'rollover_coverage': 'other'}
2026-03-11 19:24:56,459 - INFO - sn_rating_v3 - Ps Corp-Quant: altman_z value=3.000 score=100.0 family=altman
2026-03-11 19:24:56,461 - INFO - sn_rating_v3 - Ps Corp-Quant: net_debt_ebitda value=1.400 score=100.0 family=leverage
2026-03-11 19:24:56,462 - INFO - sn_rating_v3 - Ps Corp-Quant: debt_equity value=0.700 score=75.0 family=leverage
2026-03-11 19:24:56,464 - INFO - sn_rating_v3 - Ps Corp-Quant: debt_capita

In [17]:
rows_preview = []
row = 1

qual_index = list(df_qual.index)
max_rows = max(len(ratio_rows), len(qual_index))

for i in range(max_rows):
    row_dict = {"excel_row": row}

    if i < len(ratio_rows):
        ratio, _ = ratio_rows[i]
        row_dict["A"] = ratio
        row_dict["B"] = df_fin.loc[ratio].get(YEAR_T0)
        row_dict["C"] = df_fin.loc[ratio].get(YEAR_T1)

    if i < len(qual_index):
        factor = qual_index[i]
        row_dict["D"] = factor
        row_dict["E"] = df_qual.loc[factor].get(YEAR_T0)
        row_dict["F"] = df_qual.loc[factor].get(YEAR_T1)

    rows_preview.append(row_dict)
    row += 1

import pandas as pd
pd.DataFrame(rows_preview)


,excel_row,A,B,C,D,E,F
0,1,altman_z,3.00,2.400,industry_risk,NaN,NaN
1,2,net_debt_ebitda,1.40,3.100,market_position,3.0,3.0
2,3,debt_equity,0.70,1.600,revenue_diversification,3.0,3.0
3,4,debt_capital,0.40,0.570,revenue_stability,3.0,3.0
4,5,ffo_debt,0.80,0.170,business_model_resilience,3.0,3.0
5,6,interest_coverage,2.00,1.200,management_quality,3.0,3.0
6,7,fixed_charge_coverage,2.00,1.600,governance,3.0,3.0
7,8,dscr,0.80,1.050,financial_policy,3.0,3.0
8,9,ebitda_margin,0.10,0.165,sovereign_risk,3.0,3.0
9,10,ebit_margin,0.09,0.110,legal_environment,3.0,3.0
